In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import * 
from delta.tables import DeltaTable 
df_raw = spark.read.option("header",True).option("inferSchema",True).csv("/Volumes/workspace/data_files/csv_files/sales_data_dirty.csv")
df_raw.show() 


In [0]:
  #DATA CLEANING
df_cleaned = df_raw.withColumn("order_date",to_date(col("order_date"),"yyyy-MM-dd")) \
    .withColumn("price",df_raw["price"].cast(DoubleType())) \
    .withColumn("quantity",df_raw["quantity"].cast(IntegerType())) \
    .withColumn("product_name",trim(col("product_name"))) \
    .withColumn("category",trim(col("category"))) \
    .withColumn("payment_method",when(col("payment_method").isNull(),"UNKNOWN").otherwise(col("payment_method"))) \
    .withColumn("region",when(col("region").isNull(),"UNKNOWN").otherwise(col("region"))) \

print("the cleaned data : ")
df_cleaned.show()


In [0]:

# DATA VALIDATION
df_bad = df_cleaned.filter(
    (col("price") <= 0) |
    (col("quantity") <= 0) |
    (col("order_date") > current_date()) |
    (col("price").isNull()) |
    (col("quantity").isNull())
)
df_good = df_cleaned.subtract(df_bad).dropDuplicates(["order_id"]).dropna(subset=["order_id","price","quantity","order_date","region"])

df_good.show()


In [0]:

#DATA TRANSFORMATION - PHASE 1
df_transformed1 = df_good.withColumn("total_amount", col("price")* col("quantity")) \
    .withColumn("year",year(col("order_date"))) \
    .withColumn("month",month(col("order_date"))) \
    .withColumn("day",day(col("order_date"))) \
    .withColumn("year_month",concat_ws("-",col("year"),col("month"))) \
    .withColumn("quarter",quarter(col("order_date")))
   
    

print("FINAL PHASE 1 TRANSFORMED DATA:")
df_transformed1.show()

In [0]:

#DATA TRANSFORMATION - PHASE 2
#UDF for sales_category
def sales_cat(salary):
    if salary > 100000:
        return "high"
    elif salary >=20000:
        return  "medium"
    else:
        return "low"

cat_udf = udf(sales_cat,StringType())

df_transformed2 = df_transformed1.withColumn("sales_category",cat_udf(col("total_amount"))) \
    .withColumn("bulk_flag",when(col("quantity")>=5,"YES").otherwise("NO")) \
    .withColumn("day_type", when((date_format(col("order_date"),"EEEE") == "Saturday") \
                                 | (date_format(col("order_date"),"EEEE") =="Sunday"),"WeekEnd") \
                                 .otherwise("WeekDay")) \
    .withColumn("region_type", when((col("region").isin(["North","South"])),"Domestic").otherwise("International")) \


df_transformed2.show()

df_transformed2.write.mode("append").parquet("/Volumes/workspace/data_files/csv_files/sales_transformed.parquet")

In [0]:
# ==========================================================
# Imports
# ==========================================================

from delta.tables import DeltaTable
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DateType,
    BooleanType
)
from pyspark.sql.functions import current_date, lit
from datetime import date

# ==========================================================
# 1️⃣ Initial Target Data (SCD Type 1)
# ==========================================================

initial_data = [
    (1, "John", "Chennai"),
    (2, "Sam", "Mumbai"),
    (3, "Raj", "Delhi")
]

columns = ["customer_id", "name", "city"]

target_df = spark.createDataFrame(initial_data, columns)

print("the initial data :")

target_df.show()

target_path = "/Volumes/workspace/data_files/csv_files/customer_dim"

target_df.write.format("delta").mode("overwrite").save(target_path)

# ==========================================================
# 2️⃣ Incoming Source Data
# ==========================================================
columns = ["customer_id", "name", "city"]

source_data = [
    (1, "John", "Bangalore"),   # City changed
    (4, "Meena", "Hyderabad")   # New record
]

source_df = spark.createDataFrame(source_data, columns)

# ==========================================================
# ================== SCD TYPE 1 =============================
# ==========================================================

print("Running SCD Type 1...")

delta_table = DeltaTable.forPath(spark, target_path)

#source_df.write.mode("overwrite").save("\path_contains_the_initial_data_table")

delta_table.alias("t").merge(
    source_df.alias("s"),
    "t.customer_id = s.customer_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("SCD Type 1 Completed")
spark.read.format("delta").load(target_path).show()

# ==========================================================
# ================== SCD TYPE 2 =============================
# ==========================================================

print("Running SCD Type 2...")

# Explicit Schema (MANDATORY)
schema_scd2 = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("start_date", DateType(), True),
    StructField("end_date", DateType(), True),
    StructField("is_current", BooleanType(), True)
])

target_scd2_data = [
    (1, "John", "Chennai", date(2026, 1, 1), None, True),
    (2, "Sam", "Mumbai", date(2026, 1, 1), None, True),
]

target_scd2_df = spark.createDataFrame(target_scd2_data, schema_scd2)

target_scd2_path = "/Volumes/workspace/data_files/csv_files/customer_dimension_scd2"

target_scd2_df.write.format("delta").mode("overwrite").save(target_scd2_path)

delta_table_scd2 = DeltaTable.forPath(spark, target_scd2_path)

# Add SCD2 columns to source
source_scd2_df = (
    source_df
    .withColumn("start_date", current_date())
    .withColumn("end_date", lit(None).cast(DateType()))
    .withColumn("is_current", lit(True))
)

# Step 1: Expire old record if city changed
delta_table_scd2.alias("t").merge(
    source_scd2_df.alias("s"),
    "t.customer_id = s.customer_id AND t.is_current = true"
).whenMatchedUpdate(
    condition="t.city <> s.city",
    set={
        "end_date": "current_date()",
        "is_current": "false"
    }
).execute()

delta_table_scd2.alias("t").merge(
    source_scd2_df.alias("s"),
    "t.customer_id = s.customer_id AND t.is_current = true"
).whenNotMatchedInsert(
    values={
        "customer_id": "s.customer_id",
        "name": "s.name",
        "city": "s.city",
        "start_date": "s.start_date",
        "end_date": "NULL",
        "is_current": "true"
    }
).execute()

print("SCD Type 2 Completed")
spark.read.format("delta").load(target_scd2_path).show()

# ==========================================================
# ================== SCD TYPE 3 =============================
# ==========================================================

print("Running SCD Type 3...")

schema_scd3 = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("previous_city", StringType(), True)
])

target_scd3_data = [
    (1, "John", "Chennai", None),
    (2, "Sam", "Mumbai", None)
]

target_scd3_df = spark.createDataFrame(target_scd3_data, schema_scd3)

target_scd3_path = "/Volumes/workspace/data_files/csv_files/customer_dimension_scd3"

target_scd3_df.write.format("delta").mode("overwrite").save(target_scd3_path)

delta_table_scd3 = DeltaTable.forPath(spark, target_scd3_path)

delta_table_scd3.alias("t").merge(
    source_df.alias("s"),
    "t.customer_id = s.customer_id"
).whenMatchedUpdate(
    set={
        "previous_city": "t.city",
        "city": "s.city"
    }
).whenNotMatchedInsert(
    values={
        "customer_id": "s.customer_id",
        "name": "s.name",
        "city": "s.city",
        "previous_city": "NULL"
    }
).execute()

print("SCD Type 3 Completed")
spark.read.format("delta").load(target_scd3_path).show()

print("Running SCD Type 4...")

# -----------------------------
# Type 4 Concept:
# Separate history table
# -----------------------------

# Main current table
schema_scd4_main = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True)
])

main_data = [
    (1, "John", "Chennai"),
    (2, "Sam", "Mumbai")
]

main_path = "/Volumes/workspace/data_files/csv_files/customer_dimension_scd4_main"

main_df = spark.createDataFrame(main_data, schema_scd4_main)
main_df.write.format("delta").mode("overwrite").save(main_path)

# History table
schema_scd4_history = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("change_date", DateType(), True)
])

history_path = "/Volumes/workspace/data_files/csv_files/customer_dimension_scd4_history"

empty_history_df = spark.createDataFrame([], schema_scd4_history)
empty_history_df.write.format("delta").mode("overwrite").save(history_path)

main_delta = DeltaTable.forPath(spark, main_path)
history_delta = DeltaTable.forPath(spark, history_path)

# Step 1: Insert old records into history if changed
changed_records = (
    main_delta.toDF().alias("t")
    .join(source_df.alias("s"), "customer_id")
    .where("t.city <> s.city")
    .selectExpr(
        "t.customer_id",
        "t.name",
        "t.city",
        "current_date() as change_date"
    )
)

changed_records.write.format("delta").mode("append").save(history_path)

# Step 2: Update main table (Type 1 behavior)
main_delta.alias("t").merge(
    source_df.alias("s"),
    "t.customer_id = s.customer_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("SCD Type 4 Completed")

spark.read.format("delta").load(main_path).show()
spark.read.format("delta").load(history_path).show()

print("Running SCD Type 6...")

# -----------------------------
# Type 6 = Type 1 + Type 2 + Type 3
# -----------------------------

schema_scd6 = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("previous_city", StringType(), True),
    StructField("start_date", DateType(), True),
    StructField("end_date", DateType(), True),
    StructField("is_current", BooleanType(), True)
])

scd6_data = [
    (1, "John", "Chennai", None, date(2026, 1, 1), None, True),
    (2, "Sam", "Mumbai", None, date(2026, 1, 1), None, True)
]

scd6_path = "/Volumes/workspace/data_files/csv_files/customer_dimension_scd6"

scd6_df = spark.createDataFrame(scd6_data, schema_scd6)
scd6_df.write.format("delta").mode("overwrite").save(scd6_path)

delta_scd6 = DeltaTable.forPath(spark, scd6_path)

# Prepare source with SCD6 fields
source_scd6 = (
    source_df
    .withColumn("start_date", current_date())
    .withColumn("end_date", lit(None).cast(DateType()))
    .withColumn("is_current", lit(True))
)

# Step 1: Expire old record (Type 2 logic)
delta_scd6.alias("t").merge(
    source_scd6.alias("s"),
    "t.customer_id = s.customer_id AND t.is_current = true"
).whenMatchedUpdate(
    condition="t.city <> s.city",
    set={
        "end_date": "current_date()",
        "is_current": "false"
    }
).execute()

# Step 2: Insert new version (Type 2 + Type 3 logic)
delta_scd6.alias("t").merge(
    source_scd6.alias("s"),
    "t.customer_id = s.customer_id AND t.is_current = false"
).whenNotMatchedInsert(
    values={
        "customer_id": "s.customer_id",
        "name": "s.name",
        "city": "s.city",
        "previous_city": "s.city",   # Type 3 effect
        "start_date": "s.start_date",
        "end_date": "NULL",
        "is_current": "true"
    }
).execute()

print("SCD Type 6 Completed")

spark.read.format("delta").load(scd6_path).show()

